# 🧠 Entrenamiento del Modelo Base (Dummy) para Pruebas E2E

Este notebook simula el trabajo del equipo de **Data Science**. Su propósito es generar un modelo de Machine Learning sencillo utilizando **Scikit-Learn** en un entorno centralizado.

El objetivo de esta fase no es la optimización hiperparamétrica, sino **generar el artefacto físico (`.pkl`)** que nuestro pipeline de PySpark Structured Streaming consumirá posteriormente para realizar la inferencia distribuida mediante Pandas UDFs.

In [1]:
import os
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification

### 1. Generación de Datos Sintéticos
Al no disponer del conjunto de datos completo en este entorno, simularemos un entorno transaccional bancario con un fuerte desbalanceo de clases (**90% lícito, 10% fraude**), imitando la escasez de anomalías real.

In [2]:
# Definimos las 29 variables exactas esperadas por nuestro predictor.py y config.yaml
features = [
    "V1", "V2", "V3", "V4", "V5", "V6", "V7", "V8", "V9", "V10",
    "V11", "V12", "V13", "V14", "V15", "V16", "V17", "V18", "V19", "V20",
    "V21", "V22", "V23", "V24", "V25", "V26", "V27", "V28", 
    "Amount"
]

print("Generando matriz matemática sintética...")
X, y = make_classification(
    n_samples=5000,
    n_features=len(features),
    n_informative=4,
    n_redundant=1,
    n_classes=2,
    weights=[0.90, 0.10], 
    random_state=42
)

# Transformamos el array de Numpy en un DataFrame de Pandas estructurado
df = pd.DataFrame(X, columns=features)
df["Target"] = y

print(f"Dimensiones del dataset generado: {df.shape}")
display(df.head())

Generando matriz matemática sintética...
Dimensiones del dataset generado: (5000, 30)


,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Target
0,1.033349,0.445748,-0.977078,-0.126952,0.832532,0.756605,-0.666431,0.352469,0.013004,0.812811,...,-1.265724,2.220037,1.228011,-0.997159,1.961208,-0.446722,1.229127,0.450269,-0.900901,0
1,0.416461,1.448486,-1.045717,-0.345848,0.894992,0.109041,1.073679,0.253278,0.096723,1.079682,...,-3.722695,-0.739746,1.846738,-2.010457,-1.815582,-1.753026,0.162317,-0.165706,-0.200999,0
2,0.147611,0.137059,-0.874362,0.228162,-2.200158,-0.522976,0.185947,-0.718608,-0.425571,1.124142,...,-1.173809,0.602690,-2.025648,0.695907,0.542441,-0.946439,1.347017,0.236886,2.021575,1
3,0.896096,0.239909,0.476120,-0.573853,0.706481,0.091914,1.112865,0.290155,-0.087746,-0.070324,...,-2.931319,-0.516124,0.931607,-1.309194,-0.526319,0.073347,-0.163732,-2.168026,0.603936,0
4,0.719240,-2.058135,2.846098,1.724063,-1.247333,0.249024,-1.129583,0.563208,0.664790,0.822679,...,-0.170663,0.838314,1.126298,0.738270,1.737869,-0.893737,1.375542,-0.629372,1.933283,0


### 2. Entrenamiento del Modelo Centralizado
Entrenamos un modelo tipo *Ensemble* (Random Forest) ligero. Al haber ajustado el tamaño de la muestra, la operación se completa en la memoria RAM principal sin problemas.

In [3]:
print("Entrenando el algoritmo RandomForestClassifier...")
model = RandomForestClassifier(n_estimators=15, max_depth=5, random_state=42)

# Alimentamos al modelo aislando las variables predictoras (X) de la variable objetivo (y)
model.fit(df[features], df["Target"])

print("¡Modelo matemático ajustado con éxito!")

Entrenando el algoritmo RandomForestClassifier...
¡Modelo matemático ajustado con éxito!


### 3. Empaquetado y Exportación (Persistencia)
Guardamos el modelo en la ruta estandarizada del proyecto (`/src/models`) utilizando la librería `joblib`. Este paso es el puente entre la fase de experimentación y la fase productiva.

In [4]:
model_dir = "/home/jovyan/work/src/models"

# Creamos el directorio de forma segura si no existe
os.makedirs(model_dir, exist_ok=True)

# Definimos el artefacto de salida
model_path = os.path.join(model_dir, "modelo_fraude.pkl")
joblib.dump(model, model_path)

print(f"✅ Artefacto exportado correctamente. Listo para inferencia E2E en: {model_path}")

✅ Artefacto exportado correctamente. Listo para inferencia E2E en: /home/jovyan/work/src/models/modelo_fraude.pkl
